# uqEPT quick start

This template demonstrates phase-based and complex surface-integral EPT, uncertainty-guided post-processing, visualization, and output saving. Replace the placeholder paths with co-registered 3D NumPy arrays.

> Start with a cropped volume or `n_jobs=1` when testing: full 3D SI-EPT can require substantial time and memory.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

# Allow the notebook to run whether Jupyter starts in the repository root
# or directly in the examples directory.
repo_root = Path.cwd()
if not (repo_root / 'src').is_dir():
    repo_root = repo_root.parent
if not (repo_root / 'src').is_dir():
    raise RuntimeError('Run this notebook from the uqEPT repository or its examples directory.')
sys.path.insert(0, str(repo_root))

from src import (
    phase_based_surface_integral_EPT_with_uq,
    B1_based_surface_integral_EPT_with_uq,
    anatomical_min_uncertainty_weighted_mean_filter,
    anatomical_median_filter,
)

## Load and validate inputs

Required arrays must have the same 3D shape. `PhiTR` is in radians, `B1_plus` is a magnitude image, `Ref` supplies anatomical guidance, and `ROI` is a binary reconstruction mask.

In [ ]:
data_dir = Path('/path/to/co_registered_arrays')

PhiTR = np.load(data_dir / 'PhiTR.npy')
B1_plus = np.load(data_dir / 'B1_plus_magnitude.npy')
Ref = np.load(data_dir / 'reference.npy')
ROI = np.load(data_dir / 'brain_mask.npy').astype(bool)

assert PhiTR.ndim == 3
assert PhiTR.shape == B1_plus.shape == Ref.shape == ROI.shape
assert np.isrealobj(PhiTR)
assert np.any(ROI)

# Example: 3 T proton frequency and 1 mm isotropic voxels
omega = 2 * np.pi * 128e6
h = [1e-3, 1e-3, 1e-3]

## Phase-based surface-integral EPT

This returns conductivity and its propagated standard uncertainty.

In [ ]:
sigma_phase, unc_sigma_phase = phase_based_surface_integral_EPT_with_uq(
    PhiTR=PhiTR,
    Ref=Ref,
    fit_kernel_size=[11, 11, 11],
    int_kernel_size=[15, 15, 15],
    fit_shape='cube',
    int_shape='cube',
    thresh=0.05,
    omega=omega,
    h=h,
    ROI=ROI,
    n_jobs=1,
)

## Complex B-based surface-integral EPT

The factor `1j` is required when constructing the complex field from magnitude and phase.

In [ ]:
B = np.abs(B1_plus) * np.exp(1j * PhiTR / 2.0)

sigma, epsilon_r, unc_sigma, unc_epsilon_r = B1_based_surface_integral_EPT_with_uq(
    B=B,
    Ref=Ref,
    fit_kernel_size=[11, 11, 11],
    int_kernel_size=[15, 15, 15],
    fit_shape='cube',
    int_shape='cube',
    thresh=0.05,
    omega=omega,
    h=h,
    ROI=ROI,
    n_jobs=1,
)

## Post-processing

Compare the anatomical median filter with the proposed uncertainty-guided filter.

In [ ]:
sigma_median = anatomical_median_filter(
    Im=sigma, Ref=Ref, kernel_size=[21, 21, 21],
    shape='cube', thresh=0.05, ROI=ROI, n_jobs=1,
)

sigma_proposed = anatomical_min_uncertainty_weighted_mean_filter(
    Im=sigma, Ref=Ref, uncertainty=unc_sigma,
    kernel_size=[21, 21, 21], shape='cube',
    thresh=0.05, ROI=ROI, n_jobs=1,
)

## Display one slice

Use the same property-specific limits for raw and filtered maps. Uncertainty is shown separately because it represents a standard uncertainty, not an error map.

In [ ]:
z = sigma.shape[2] // 2
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4), constrained_layout=True)
items = [
    (sigma, 'Raw conductivity'),
    (sigma_median, 'Median filtered'),
    (sigma_proposed, 'Proposed'),
    (unc_sigma, 'Standard uncertainty'),
]
for ax, (volume, title) in zip(axes, items):
    image = np.where(ROI[:, :, z], volume[:, :, z], np.nan)
    im = ax.imshow(image.T, origin='lower', cmap='viridis')
    ax.set_title(title)
    ax.axis('off')
    fig.colorbar(im, ax=ax, shrink=0.78)
plt.show()

## Save outputs

In [ ]:
output_dir = Path('uqept_outputs')
output_dir.mkdir(exist_ok=True)
np.save(output_dir / 'sigma_SI.npy', sigma)
np.save(output_dir / 'epsilon_r_SI.npy', epsilon_r)
np.save(output_dir / 'unc_sigma_SI.npy', unc_sigma)
np.save(output_dir / 'unc_epsilon_r_SI.npy', unc_epsilon_r)
np.save(output_dir / 'sigma_SI_proposed.npy', sigma_proposed)